# Mesh Creation, Smoothing, and Refinement

Note: you will need to install the brain_mesh_creation module to run this notebook. 

### Setup paths

In [2]:
from pathlib import Path
import os
import sys

# Project root: assumes notebook is run from notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent

# Main folders
SRC_DIR = PROJECT_ROOT / "src"
PACKAGE_DIR = SRC_DIR / "brain_mesh_creation"
wrk_dir = str(SRC_DIR / "dependencies")
sub_dir = str(PROJECT_ROOT / "data" / "subjects")

# Subject and output names
subject_folder = "sub0045"
output_name = "output"

# Make sure src/ is importable
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from brain_mesh_creation import bmctk
from brain_mesh_creation import mesh_utils

print("PROJECT_ROOT:", PROJECT_ROOT)
print("wrk_dir:", wrk_dir)
print("sub_dir:", sub_dir)
print("subject_folder:", subject_folder)

PROJECT_ROOT: /Users/yc4421/Library/CloudStorage/Box-Box/GTA/ReCoDE_BrainMesh/ReCoDE-brain-mesh-creation
wrk_dir: /Users/yc4421/Library/CloudStorage/Box-Box/GTA/ReCoDE_BrainMesh/ReCoDE-brain-mesh-creation/src/dependencies
sub_dir: /Users/yc4421/Library/CloudStorage/Box-Box/GTA/ReCoDE_BrainMesh/ReCoDE-brain-mesh-creation/data/subjects
subject_folder: sub0045


### Meshing parameters setup

In [9]:
# Set mesh size (in mm)
mesh_size = 3

# Voxel correction variables
# It is generally recommended not to change these unless needed
iterations = 3
blocks = 1
threshold = 0.9  # Threshold for neighbour disagreement during voxel correction

# Manual centre-of-gravity edits
# Used for generating the centre of gravity for TBI simulations
cog_id = 80000000
change_ap = 0
change_lr = 0
change_is = 0
change_cog = [change_ap, change_lr, change_is]

## Mesh creation

The segmented image is converted into mesh using a direct voxel-to-element approach, generating a fully hexahedral mesh. Hexahedral elements are selected over tetrahedra due to their superior performance in explicit simulations, specifically regarding time-step stability and resistance to volumetric locking in nearly incompressible materials such as brain tissue (Tadepalli et al. 2011; Giudice et al. 2019).

To accommodate varying computational resources and fidelity requirements, the pipeline incorporates a customizable resampling module. While the default settings maintains native MRI resolution, the pipeline allows users to define a custom target element size (e.g., 1mm, 1.5 mm or 2.0 mm) prior to mesh generation. The segmented masks are automatically resampled to this user-defined voxel size, allowing users to optimize the balance between geometric accuracy and computational cost based on specific study requirements.

### Algorithmic reconstruction of meningeal structures
Following the solid mesh generation, the meningeal layers are constructed. Standard MRI-based FE meshes frequently omit meningeal structures due to their thin geometry, which is often indistinguishable at standard MRI resolutions (Chen and Ostoja-Starzewski 2010). However, these structures have shown to play a significant role in brain biomechanics by limiting the relative motion of left and right hemispheres or constraining the upward movements of the corpus callosum (Hernandez et al. 2019; Darvishi et al. 2025). Although some studies have attempted to incorporate these structures manually in the brain model, these approaches compromise the automation which is vital for large-scale patient-specific modeling (Miller et al. 2016). To address this, our pipeline algorithmically synthesizes them based on anatomical landmarks:

- Falx: The algorithm identifies the longitudinal fissure separating left and right hemispheres. A shell layer is generated in the mid-sagittal plane, extending from the corpus callosum to the interior surface of skull.

- Tentorium: The interface between the inferior surface of the occipital/temporal lobes and the superior surface of the cerebellum is identified to generate a partition representing the tentorium.

- Pia and Dura mater: The pia mater is generated as a shell layer coincident with the outer surface of the brain parenchyma, while the dura mater is generated from the inner surface of the skull mesh.

In [10]:
%%bash -s "$sub_dir" "$subject_folder" "$output_name"

echo "looking at" "$2"
cd "$1"/"$2"
echo "Removing legacy mesh data for $2"
rm -r "$3"

looking at sub0045
Removing legacy mesh data for sub0045


In [11]:
import os

subject_path = os.path.join(sub_dir, subject_folder)
pre_model_path = os.path.join(subject_path, "pre_model.nii.gz")

print("Looking at:", subject_path)

if os.path.isdir(subject_path):

    if not os.path.isfile(pre_model_path):
        print(f"Cannot find pre_model.nii.gz at: {pre_model_path}")
        print("Please run notebook 01 first to generate pre_model.nii.gz")
    else:
        print("Starting mesh generation for", subject_folder)

        # Move into subject folder so output files are written there
        os.chdir(subject_path)
        print("Working directory:", os.getcwd())

        # Initialise model creation
        my_model = bmctk.create_model("pre_model.nii.gz", output_name, wrk_dir)

        # Create relative coordinates
        my_model.create_coordinates()

        # Resample model if necessary
        my_model.resample(mesh_size)

        # Assign missing grey matter
        my_model.gm_check()

        # Check contact of grey matter to skull
        my_model.check_contact(257, 256, 42)
        my_model.check_contact(257, 256, 3)
        my_model.check_contact(257, 256, 47)
        my_model.check_contact(257, 256, 8)

        # Check contact of CSF to void
        my_model.check_contact(0, 257, 256)

        # Check contact of CSF to white matter
        my_model.check_contact(41, 42, 256)
        my_model.check_contact(2, 3, 256)
        my_model.check_contact(46, 47, 256)
        my_model.check_contact(7, 8, 256)

        # Make corrections for voxels inside other parts
        my_model.voxel_corrections(iterations, threshold, blocks)

        # Create falx
        LR_center_coords = my_model.create_falx()
        LR_center = LR_center_coords[0]

        # Create tentorium
        my_model.create_tentorium()

        # Create mesh
        max_node = my_model.create_k_mesh()
        print("Largest node number assigned:", max_node)

        # Create set list
        my_model.set_list()

        print("Finished making model for", subject_folder)
        print()
else:
    print("Cannot see folder", subject_folder, "in", sub_dir)

Looking at: /Users/yc4421/Library/CloudStorage/Box-Box/GTA/ReCoDE_BrainMesh/ReCoDE-brain-mesh-creation/data/subjects/sub0045
Starting mesh generation for sub0045
Working directory: /Users/yc4421/Library/CloudStorage/Box-Box/GTA/ReCoDE_BrainMesh/ReCoDE-brain-mesh-creation/data/subjects/sub0045
Reading brain data... 
Assigning missing brain matter to grey matter and brain stem... 
Number of missing voxels: 258
Number of missing voxels: 225
Number of missing voxels: 214
Number of missing voxels: 208
Number of missing voxels: 206
Number of missing voxels: 206
Assigning remaining voxels... 
Number of missing voxels: 0
Voxel correction: 
Iteration 1
Values changed: 78

Voxel correction: 
Iteration 2
Values changed: 1

Voxel correction: 
Iteration 3
Values changed: 0

Falx location voxel: 30.347199870636636
Falx location scanner: 0.04159961190990202
Reading nodal coordinates... 
Assigning element nodes... 
Writting K file...
Largest node number assigned: 883872
Finished making model for sub00

## Mesh smoothing
A known limitation of direct voxel-to-element conversion is the creation of stair-step surface artifacts. These interfaces, particularly at the outer boundaries of the brain, can lead to artificial stress concentrations and numerical instabilities. To mitigate this, we applied a Laplacian smoothing algorithm adapted from the methodology of Chen and Ostoja-Starzewski (Chen and Ostoja-Starzewski 2010). The algorithm operates on the boundary surface nodes of the generated mesh through an iterative process. This process effectively removes the stair-steps shape on the boundaries, while constraining the total volume change to minimal levels. This step is vital for ensuring that the interfaces between the brain regions, meninges, CSF and skull are smooth, thereby preventing unrealistic stress concentration during simulation.

In [12]:
%%bash -s "$mesh_size" "$wrk_dir" "$sub_dir" "$subject_folder" "$output_name"

set -e

MESH_SIZE="$1"
WRK_DIR="$2"
SUB_DIR="$3"
SUBJECT="$4"
OUTPUT_NAME="$5"

SUBJECT_PATH="${SUB_DIR}/${SUBJECT}"
OUTPUT_PATH="${SUBJECT_PATH}/${OUTPUT_NAME}"
AOUT_PATH="${WRK_DIR}/rs/a.out"

echo "Looking at ${SUBJECT}"
cd "${SUBJECT_PATH}" || exit 1
pwd
echo "Moved to ${SUBJECT}"

if [ ! -f "${AOUT_PATH}" ]; then
    echo "Cannot find smoothing executable: ${AOUT_PATH}"
    exit 1
fi

if [ ! -f "${OUTPUT_PATH}/mesh.k" ]; then
    echo "Cannot find input mesh: ${OUTPUT_PATH}/mesh.k"
    exit 1
fi

cd "${OUTPUT_PATH}" || exit 1
echo "Running mesh smoothing in $(pwd)"

"${AOUT_PATH}" mesh.k mesh_smoothed.k 8 0.2 200 2000 "${MESH_SIZE}"

echo "Created: ${OUTPUT_PATH}/mesh_smoothed.k"
cd ..

Looking at sub0045
/Users/yc4421/Library/CloudStorage/Box-Box/GTA/ReCoDE_BrainMesh/ReCoDE-brain-mesh-creation/data/subjects/sub0045
Moved to sub0045
Running mesh smoothing in /Users/yc4421/Library/CloudStorage/Box-Box/GTA/ReCoDE_BrainMesh/ReCoDE-brain-mesh-creation/data/subjects/sub0045/output


bash: line 32: /Users/yc4421/Library/CloudStorage/Box-Box/GTA/ReCoDE_BrainMesh/ReCoDE-brain-mesh-creation/src/dependencies/rs/a.out: Permission denied


CalledProcessError: Command 'b'\nset -e\n\nMESH_SIZE="$1"\nWRK_DIR="$2"\nSUB_DIR="$3"\nSUBJECT="$4"\nOUTPUT_NAME="$5"\n\nSUBJECT_PATH="${SUB_DIR}/${SUBJECT}"\nOUTPUT_PATH="${SUBJECT_PATH}/${OUTPUT_NAME}"\nAOUT_PATH="${WRK_DIR}/rs/a.out"\n\necho "Looking at ${SUBJECT}"\ncd "${SUBJECT_PATH}" || exit 1\npwd\necho "Moved to ${SUBJECT}"\n\nif [ ! -f "${AOUT_PATH}" ]; then\n    echo "Cannot find smoothing executable: ${AOUT_PATH}"\n    exit 1\nfi\n\nif [ ! -f "${OUTPUT_PATH}/mesh.k" ]; then\n    echo "Cannot find input mesh: ${OUTPUT_PATH}/mesh.k"\n    exit 1\nfi\n\ncd "${OUTPUT_PATH}" || exit 1\necho "Running mesh smoothing in $(pwd)"\n\n"${AOUT_PATH}" mesh.k mesh_smoothed.k 8 0.2 200 2000 "${MESH_SIZE}"\n\necho "Created: ${OUTPUT_PATH}/mesh_smoothed.k"\ncd ..\n'' returned non-zero exit status 1.

## Mesh refinement: Brain-Skull Contact Prevention

Furthermore, to ensure there is no non-physical contact between brain tissue and the skull following the smoothing process, the pipeline includes a contact repair module. This algorithm iterates through the brain surface boundary to detect any brain nodes that are shared with the skull mesh. Any skull elements associated with these shared nodes are flagged and reclassified as CSF. This ensures a strict kinematic separation between the brain parenchyma and the skull, guaranteeing a continuous CSF layer at the brain-skull interface.


In [ ]:
import os
import sys
from pathlib import Path

# Make sure src/ is importable
PROJECT_ROOT = Path.cwd().resolve().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from brain_mesh_creation import mesh_utils

current_subject_dir = os.path.join(sub_dir, subject_folder)
mesh_output_dir = os.path.join(current_subject_dir, output_name)

input_k_file = os.path.join(mesh_output_dir, "mesh_smoothed.k")
output_k_file = os.path.join(mesh_output_dir, "mesh_smoothed_revised.k")

print(f"--- Processing Subject: {subject_folder} ---")
print("Applying brain-skull interface regularization...")
print("Input:", input_k_file)
print("Output:", output_k_file)

if not os.path.isfile(input_k_file):
    print(f"Cannot find input mesh file: {input_k_file}")
else:
    mesh_utils.create_csf_buffer(input_k_file, output_k_file)
    print("Finished creating revised mesh:")
    print(output_k_file)